# Домашнє завдання: Внесення оновлень в БД і робота з транзакціями

Це ДЗ передбачене під виконання на локальній машині. Виконання з Google Colab буде суттєво ускладнене.

## Підготовка
1. Переконайтесь, що у вас встановлены необхідні бібліотеки:
   ```bash
   pip install sqlalchemy pymysql pandas matplotlib seaborn python-dotenv
   ```

2. Створіть файл `.env` з параметрами підключення до бази даних classicmodels. Базу даних ви можете отримати через

  - docker-контейнер згідно існтрукції в [документі](https://www.notion.so/hannapylieva/Docker-1eb94835849480c9b2e7f5dc22ee4df9), також відео інструкції присутні на платформі - уроки "MySQL бази, клієнт для роботи з БД, Docker і ChatGPT для запитів" та "Як встановити Docker для роботи з базами даних без терміналу"
  - або встановивши локально цю БД - для цього перегляньте урок "Опціонально. Встановлення MySQL та  БД Сlassicmodels локально".
  
  Приклад `.env` файлу ми створювали в лекції. Ось його обовʼязкове наповнення:
    ```
    DB_HOST=your_host
    DB_PORT=3306 або 3307 - той, який Ви налаштували
    DB_USER=your_username
    DB_PASSWORD=your_password
    DB_NAME=classicmodels
    ```
  Якщо ви створили цей файл під час перегляду лекції - **новий створювати не треба**. Замініть лише назву БД, або пропишіть назву в коді створення підключення (замість отримання назви цільової БД зі змінних оточення). Але переконайтесь, що до `.env` файл лежить в тій самій папці, що і цей ноутбук.

  **УВАГА!** НЕ копіюйте скрит для **створення** `.env` файлу. В лекції він наводиться для прикладу. І давалось пояснення, що в реальних проєктах ми НІКОЛИ не пишемо доступи до бази в коді. Копіювання скрипта для створення `.env` файлу сюди в ДЗ буде вважатись грубою помилкою і ми зніматимемо бали.

3. Налаштуйте підключення через SQLAlchemy до БД за прикладом в лекції.

Рекомендую вивести (відобразити) змінну engine після створення. Вона має бути не None! Якщо None - значить у Вас не підтягнулись налаштування з .env файла.

Ви також можете налаштувати параметри підключення до БД без .env файла, просто прописавши текстом в відповідних місцях. Це - не рекомендований підхід.




In [1]:
pip install sqlalchemy pymysql pandas matplotlib seaborn python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [9]:
import datetime
import requests
import json
import os

from dotenv import load_dotenv
import pandas as pd
import sqlalchemy as sa
from sqlalchemy import create_engine, text, MetaData, Table
from sqlalchemy.orm import sessionmaker

In [11]:
# Створюємо .env файл з параметрами підключення
env_content = """
# Database Configuration
DB_HOST=127.0.0.1
DB_PORT=3307
DB_USER=root
DB_PASSWORD=root
DB_NAME=employees

# API Keys
EXCHANGE_API_KEY=your_api_key_here

# Environment
ENV=development
"""

with open('.env', 'w') as f:
    f.write(env_content)

print("✅ Файл .env створено")
print("🔐 УВАГА: Додайте .env до .gitignore!")

✅ Файл .env створено
🔐 УВАГА: Додайте .env до .gitignore!


In [16]:
load_dotenv(override=True)

user = os.getenv('DB_USER')
password = os.getenv('DB_PASSWORD')
host = os.getenv('DB_HOST')
port = os.getenv('DB_PORT')

db = "classicmodels" 

connection_string = f"mysql+pymysql://{user}:{password}@{host}:{port}/{db}"
engine = create_engine(connection_string)

with engine.connect() as conn:
    current_db = conn.execute(text("SELECT DATABASE()")).scalar()
    print(f"✅ Зараз підключено до бази: {current_db}")

✅ Зараз підключено до бази: classicmodels


## Завдання

### Завдання 1: Оновлення інформації про клієнта (2 бали)

**Створіть функцію для оновлення контактної інформації клієнта за його номером** з наступними можливостями:
- Оновлення телефону клієнта
- Оновлення email (якщо поле існує в таблиці)

Опціонально, якщо вам хочеться більше практики:
- Логування змін в окрему таблицю

Використайте підхід з параметризованими запитами через `text()` та `UPDATE` оператор. Не забудьте на початку перевірити чи існує клієнт з таким номером в базі - це хороша практика.

Отримати всі колонки, які існують в таблиці ви можете наступним запитом
```
  SELECT COLUMN_NAME, DATA_TYPE
  FROM INFORMATION_SCHEMA.COLUMNS
  WHERE TABLE_NAME = 'customers'
```

Запустіть функцію і продемонструйте її роботу, запустивши SELECT, який допоможе це зробити.



In [18]:
import pandas as pd
from sqlalchemy import text

# ---------- 0. Перевірка, що ми в правильній базі ----------
with engine.connect() as conn:
    print("Поточна база:", conn.execute(text("SELECT DATABASE()")).scalar())

# ---------- 1. Колонки таблиці customers ----------
columns_query = text("""
    SELECT COLUMN_NAME, DATA_TYPE
    FROM INFORMATION_SCHEMA.COLUMNS
    WHERE TABLE_SCHEMA = DATABASE()
      AND TABLE_NAME = 'customers'
""")
display(pd.read_sql(columns_query, engine))

# ---------- 2. Таблиця для логування ----------
with engine.begin() as conn:
    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS customer_changes_log (
            logId INT AUTO_INCREMENT PRIMARY KEY,
            customerNumber INT NOT NULL,
            fieldName VARCHAR(50) NOT NULL,
            oldValue VARCHAR(255),
            newValue VARCHAR(255),
            changedAt TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    """))

# ---------- 3. Функція оновлення ----------
def update_customer_contact(engine, customer_number, phone=None, email=None):
    """Оновлює телефон та/або email клієнта за номером, з логуванням змін."""
    if phone is None and email is None:
        return {"success": False, "message": "Не вказано, що оновлювати."}

    with engine.begin() as conn:  # транзакція: commit / rollback автоматично

        # Перевірка, чи існує клієнт
        exists = conn.execute(
            text("SELECT COUNT(*) FROM customers WHERE customerNumber = :num"),
            {"num": customer_number}
        ).scalar()
        if not exists:
            return {"success": False,
                    "message": f"Клієнт з номером {customer_number} не знайдений."}

        # Перевірка, чи є колонка email
        email_exists = conn.execute(text("""
            SELECT COUNT(*) FROM INFORMATION_SCHEMA.COLUMNS
            WHERE TABLE_SCHEMA = DATABASE()
              AND TABLE_NAME = 'customers'
              AND COLUMN_NAME = 'email'
        """)).scalar() > 0

        # Що оновлюємо: {назва колонки: нове значення}
        updates = {}
        skipped = []
        if phone is not None:
            updates["phone"] = phone
        if email is not None:
            if email_exists:
                updates["email"] = email
            else:
                skipped.append("email (у таблиці немає такої колонки)")

        if not updates:
            return {"success": False,
                    "message": "Нічого оновлювати: " + ", ".join(skipped)}

        # Старі значення (для логу). Назви колонок беруться лише з нашого словника.
        old_row = conn.execute(
            text(f"SELECT {', '.join(updates)} FROM customers WHERE customerNumber = :num"),
            {"num": customer_number}
        ).mappings().one()

        # Запис у лог
        for field, new_value in updates.items():
            conn.execute(text("""
                INSERT INTO customer_changes_log
                    (customerNumber, fieldName, oldValue, newValue)
                VALUES (:num, :field, :old, :new)
            """), {"num": customer_number, "field": field,
                   "old": old_row[field], "new": new_value})

        # UPDATE із параметризованими значеннями
        set_clause = ", ".join(f"{col} = :{col}" for col in updates)
        result = conn.execute(
            text(f"UPDATE customers SET {set_clause} WHERE customerNumber = :num"),
            {**updates, "num": customer_number}
        )

    message = f"Клієнта {customer_number} оновлено. Змінено рядків: {result.rowcount}."
    if skipped:
        message += " Пропущено: " + ", ".join(skipped) + "."
    return {"success": True, "message": message}

# ---------- 4. Демонстрація ----------
CUSTOMER_ID = 103

show_query = text("""
    SELECT customerNumber, customerName, contactFirstName, contactLastName, phone
    FROM customers
    WHERE customerNumber = :num
""")

print("\nДО оновлення:")
display(pd.read_sql(show_query, engine, params={"num": CUSTOMER_ID}))

print(update_customer_contact(engine, CUSTOMER_ID,
                              phone="+380 50 123 4567",
                              email="test@example.com"))

print("\nПІСЛЯ оновлення:")
display(pd.read_sql(show_query, engine, params={"num": CUSTOMER_ID}))

print("\nПеревірка неіснуючого клієнта:")
print(update_customer_contact(engine, 999999, phone="000"))

print("\nЛог змін:")
display(pd.read_sql(text("SELECT * FROM customer_changes_log ORDER BY logId DESC"), engine))

Поточна база: classicmodels


,COLUMN_NAME,DATA_TYPE
0,customerNumber,int
1,customerName,varchar
2,contactLastName,varchar
3,contactFirstName,varchar
4,phone,varchar
5,addressLine1,varchar
6,addressLine2,varchar
7,city,varchar
8,state,varchar
9,postalCode,varchar



ДО оновлення:


,customerNumber,customerName,contactFirstName,contactLastName,phone
0,103,Atelier graphique,Carine,Schmitt,+380 50 123 4567


{'success': True, 'message': 'Клієнта 103 оновлено. Змінено рядків: 1. Пропущено: email (у таблиці немає такої колонки).'}

ПІСЛЯ оновлення:


,customerNumber,customerName,contactFirstName,contactLastName,phone
0,103,Atelier graphique,Carine,Schmitt,+380 50 123 4567



Перевірка неіснуючого клієнта:
{'success': False, 'message': 'Клієнт з номером 999999 не знайдений.'}

Лог змін:


,logId,customerNumber,fieldName,oldValue,newValue,changedAt
0,2,103,phone,+380 50 123 4567,+380 50 123 4567,2026-11-22 11:43:08
1,1,103,phone,40.32.2555,+380 50 123 4567,2026-11-22 11:42:35


### Завдання 2: Створення нового замовлення з транзакцією (5 балів)

**Реалізуйте процес створення нового замовлення** з наступними кроками в одній транзакції:
- Створення запису в таблиці `orders`
- Додавання товарних позицій в `orderdetails`
- Перевірка наявності товарів на складі
- Зменшення кількості товарів на складі

Запустіть процес з тестовими даними і продемонструйте через SELECT, що процес успішно відпрацював і були виконані необхідні операції.




In [22]:
import pandas as pd
from datetime import date, timedelta
from sqlalchemy import text, bindparam

# ---------- 0. Перевірка бази ----------
with engine.connect() as conn:
    db = conn.execute(text("SELECT DATABASE()")).scalar()
print("Поточна база:", db)
assert db == "classicmodels", "engine підключений не до classicmodels"


# ---------- 1. Функція створення замовлення ----------
class OrderError(Exception):
    """Бізнес-помилка: замовлення не можна створити."""


def create_order(engine, customer_number, items, comments=None):
    if not items:
        return {"success": False, "message": "Список товарів порожній."}
    codes = [i["productCode"] for i in items]
    if len(codes) != len(set(codes)):
        return {"success": False, "message": "У замовленні є дублікати productCode."}
    if any(i["quantity"] <= 0 for i in items):
        return {"success": False, "message": "Кількість має бути більшою за 0."}

    try:
        with engine.begin() as conn:  # commit при успіху, rollback при будь-якому виключенні

            # 1. Перевірка клієнта
            exists = conn.execute(
                text("SELECT COUNT(*) FROM customers WHERE customerNumber = :c"),
                {"c": customer_number}
            ).scalar()
            if not exists:
                raise OrderError(f"Клієнта {customer_number} не знайдено.")

            # 2. Перевірка наявності товарів на складі
            stock_query = text("""
                SELECT productCode, quantityInStock, MSRP
                FROM products
                WHERE productCode IN :codes
                FOR UPDATE
            """).bindparams(bindparam("codes", expanding=True))
            stock = {r["productCode"]: r for r in
                     conn.execute(stock_query, {"codes": codes}).mappings().all()}

            for item in items:
                code, qty = item["productCode"], item["quantity"]
                if code not in stock:
                    raise OrderError(f"Товар {code} не існує.")
                if stock[code]["quantityInStock"] < qty:
                    raise OrderError(
                        f"Недостатньо товару {code} на складі: "
                        f"потрібно {qty}, є {stock[code]['quantityInStock']}."
                    )

            # 3. Створення запису в orders
            order_number = conn.execute(
                text("SELECT COALESCE(MAX(orderNumber), 0) + 1 FROM orders FOR UPDATE")
            ).scalar()

            conn.execute(text("""
                INSERT INTO orders
                    (orderNumber, orderDate, requiredDate, shippedDate, status, comments, customerNumber)
                VALUES
                    (:num, :odate, :rdate, NULL, 'In Process', :comments, :cust)
            """), {
                "num": order_number,
                "odate": date.today(),
                "rdate": date.today() + timedelta(days=7),
                "comments": comments,
                "cust": customer_number,
            })

            # 4. Додавання позицій у orderdetails + 5. зменшення складу
            for line_no, item in enumerate(items, start=1):
                code, qty = item["productCode"], item["quantity"]
                price = item.get("priceEach", stock[code]["MSRP"])

                conn.execute(text("""
                    INSERT INTO orderdetails
                        (orderNumber, productCode, quantityOrdered, priceEach, orderLineNumber)
                    VALUES (:num, :code, :qty, :price, :line)
                """), {"num": order_number, "code": code, "qty": qty,
                       "price": price, "line": line_no})

                upd = conn.execute(text("""
                    UPDATE products
                    SET quantityInStock = quantityInStock - :qty
                    WHERE productCode = :code AND quantityInStock >= :qty
                """), {"qty": qty, "code": code})
                if upd.rowcount != 1:
                    raise OrderError(f"Не вдалося зменшити склад для {code}.")

        return {"success": True, "orderNumber": order_number,
                "message": f"Замовлення {order_number} створено."}

    except OrderError as e:
        return {"success": False, "message": f"Відкочено: {e}"}


# ---------- 2. Допоміжні SELECT ----------
def show_stock(codes):
    q = text("""
        SELECT productCode, productName, quantityInStock
        FROM products WHERE productCode IN :codes ORDER BY productCode
    """).bindparams(bindparam("codes", expanding=True))
    return pd.read_sql(q, engine, params={"codes": codes})


def show_order(order_number):
    q = text("""
        SELECT o.orderNumber, o.orderDate, o.status, o.customerNumber, o.comments,
               d.orderLineNumber, d.productCode, d.quantityOrdered, d.priceEach
        FROM orders o
        JOIN orderdetails d ON d.orderNumber = o.orderNumber
        WHERE o.orderNumber = :num
        ORDER BY d.orderLineNumber
    """)
    return pd.read_sql(q, engine, params={"num": order_number})


# ---------- 3. Демонстрація: успішне замовлення ----------
CUSTOMER_ID = 103
test_items = [
    {"productCode": "S10_1678", "quantity": 10},
    {"productCode": "S10_1949", "quantity": 5},
]
test_codes = [i["productCode"] for i in test_items]

print("\nСклад ДО замовлення:")
display(show_stock(test_codes))

result = create_order(engine, CUSTOMER_ID, test_items, comments="Test order (homework)")
print("\nРезультат:", result)

if result["success"]:
    print("\nСтворене замовлення (orders + orderdetails):")
    display(show_order(result["orderNumber"]))

print("\nСклад ПІСЛЯ замовлення (має зменшитись на 10 і 5):")
display(show_stock(test_codes))


# ---------- 4. Демонстрація: відкат транзакції ----------
with engine.connect() as conn:
    orders_before = conn.execute(text("SELECT COUNT(*) FROM orders")).scalar()

bad_items = [
    {"productCode": "S10_1678", "quantity": 1},       # сам по собі валідний
    {"productCode": "S10_1949", "quantity": 10**6},   # забагато, викличе відкат
]
bad_result = create_order(engine, CUSTOMER_ID, bad_items)
print("\nНеуспішне замовлення:", bad_result)

with engine.connect() as conn:
    orders_after = conn.execute(text("SELECT COUNT(*) FROM orders")).scalar()
print(f"Кількість замовлень до/після: {orders_before} / {orders_after} (має бути однаково)")

print("\nСклад після невдалої спроби (не змінився):")
display(show_stock(test_codes))

Поточна база: classicmodels

Склад ДО замовлення:


,productCode,productName,quantityInStock
0,S10_1678,1969 Harley Davidson Ultimate Chopper,7933
1,S10_1949,1952 Alpine Renault 1300,7305



Результат: {'success': True, 'orderNumber': 10426, 'message': 'Замовлення 10426 створено.'}

Створене замовлення (orders + orderdetails):


,orderNumber,orderDate,status,customerNumber,comments,orderLineNumber,productCode,quantityOrdered,priceEach
0,10426,2026-11-22,In Process,103,Test order (homework),1,S10_1678,10,95.7
1,10426,2026-11-22,In Process,103,Test order (homework),2,S10_1949,5,214.3



Склад ПІСЛЯ замовлення (має зменшитись на 10 і 5):


,productCode,productName,quantityInStock
0,S10_1678,1969 Harley Davidson Ultimate Chopper,7923
1,S10_1949,1952 Alpine Renault 1300,7300



Неуспішне замовлення: {'success': False, 'message': 'Відкочено: Недостатньо товару S10_1949 на складі: потрібно 1000000, є 7300.'}
Кількість замовлень до/після: 327 / 327 (має бути однаково)

Склад після невдалої спроби (не змінився):


,productCode,productName,quantityInStock
0,S10_1678,1969 Harley Davidson Ultimate Chopper,7923
1,S10_1949,1952 Alpine Renault 1300,7300
